# Analyse Exploratoire des Données (EDA) du Positionnement Salarial

**Objet du document :**
Ce notebook présente notre démarche d'analyse exploratoire des données salariales et RH en vue de construire un modèle prédictif de la rémunération fixe annuelle (`Fixe_Annuel_MAD`).
Ce support de travail a pour vocation de nous permettre d'examiner ensemble la structure actuelle des rémunérations, de valider les hypothèses métier et d'arrêter conjointement le périmètre des variables explicatives qui alimenteront le modèle de régression.

**Structure de lecture :**
Chaque étape analytique est précédée d'une synthèse méthodologique structurée autour de trois axes :
1. **Objectif analytique :** Ce que la visualisation met en évidence d'un point de vue RH et statistique.
2. **Grille de lecture :** Les éléments clés à observer sur nos données réelles.
3. **Arbitrage de modélisation :** La décision technique et métier à valider pour la construction de la régression.

---

## Étape 0 : Consolidation des Sources, Dédoublonnage et Contrôles d'Intégrité

Cette étape préliminaire assure la fiabilité de la base d'étude à partir des trois référentiels (`employes.xlsx`, `bands.xlsx`, `market.xlsx`) :
1. Application de la règle de gestion sur les identifiants (`Matricule`) : conservation exclusive de la première occurrence en cas de doublon.
2. Croisement avec les grilles internes et les médianes de marché par couple (`Job_Family`, `Grade`).
3. Contrôle d'intégrité des données (cohérence entre l'âge et l'ancienneté, unicité de l'indicateur de tension `Hot_job` par poste `Job_Title`) et calcul des ratios standards de rémunération.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import gaussian_kde
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# 1. Chargement des référentiels
# Les trois fichiers sont cherches dans le dossier courant (Colab : fichiers deposes a cote du notebook),
# puis dans input/ du projet (lancer_notebook.bat ouvre le notebook depuis notebooks/)
dossiers_possibles = [Path.cwd(), Path.cwd() / 'input', *[p / 'input' for p in Path.cwd().parents]]
DOSSIER_DONNEES = next((d for d in dossiers_possibles if (d / 'employes.xlsx').is_file()), None)
if DOSSIER_DONNEES is None:
    raise FileNotFoundError(f"employes.xlsx introuvable : ni dans {Path.cwd()}, ni dans un dossier input/ du projet.")
print(f"Fichiers lus dans : {DOSSIER_DONNEES}")

df_emp = pd.read_excel(DOSSIER_DONNEES / 'employes.xlsx')
df_bands = pd.read_excel(DOSSIER_DONNEES / 'bands.xlsx')[['Job_Family', 'Grade', 'Min', 'Mid', 'Max']]
df_market = pd.read_excel(DOSSIER_DONNEES / 'market.xlsx')[['Job_Family', 'Grade', 'Median']]

# 2. Dédoublonnage sur le Matricule (conservation de la première ligne)
nb_initial = len(df_emp)
df_emp = df_emp.drop_duplicates(subset=['Matricule'], keep='first').copy()
print(f"Controle dedoublonnage Matricule : {nb_initial} lignes initiales -> {len(df_emp)} lignes retenues ({nb_initial - len(df_emp)} doublons ecartes).")

# 3. Jointures avec les grilles internes (bands) et les donnees de marche (market)
df = df_emp.merge(df_bands, on=['Job_Family', 'Grade'], how='left')
df = df.merge(df_market, on=['Job_Family', 'Grade'], how='left')

# 4. Audit d'integrite des donnees
print("\n--- AUDIT D'INTEGRITE DES DONNEES ---")
print(f"Collaborateurs sans correspondance de grille interne (Mid manquant) : {df['Mid'].isna().sum()}")
print(f"Collaborateurs sans correspondance de marche (Median manquant)       : {df['Median'].isna().sum()}")

df['Age_Embauche'] = df['Age'] - df['Anciennete']
anomalies_age = df[df['Age_Embauche'] < 18]
print(f"Anomalies de coherence Age/Anciennete (Age d'embauche < 18 ans)     : {len(anomalies_age)}")

hotjob_par_poste = df.groupby('Job_Title')['Hot_job'].nunique()
postes_incoherents = hotjob_par_poste[hotjob_par_poste > 1]
print(f"Postes (Job_Title) presentant plusieurs valeurs de Hot_job          : {len(postes_incoherents)}")

# 5. Construction des indicateurs metier
# Note methodologique : Compa_Ratio, Range_Penetration et Market_Ratio intègrent le salaire réel dans leur calcul.
df['Compa_Ratio'] = df['Fixe_Annuel_MAD'] / df['Mid']
df['Range_Penetration'] = (df['Fixe_Annuel_MAD'] - df['Min']) / (df['Max'] - df['Min'])
df['Market_Ratio'] = df['Fixe_Annuel_MAD'] / df['Median']

# Variables structurelles mobilisables comme regresseurs (independantes du salaire individuel)
df['Ratio_Grille_Marche'] = df['Mid'] / df['Median']
df['Amplitude_Grille'] = (df['Max'] - df['Min']) / df['Mid']

df.head()

Controle dedoublonnage Matricule : 10010 lignes initiales -> 10000 lignes retenues (10 doublons ecartes).

--- AUDIT D'INTEGRITE DES DONNEES ---
Collaborateurs sans correspondance de grille interne (Mid manquant) : 0
Collaborateurs sans correspondance de marche (Median manquant)       : 0
Anomalies de coherence Age/Anciennete (Age d'embauche < 18 ans)     : 0
Postes (Job_Title) presentant plusieurs valeurs de Hot_job          : 0


,Matricule,Nom,Entite_N1,Entite_N2,Pays,Job_Family,Job_Title,Grade,Fixe_Annuel_MAD,Age,...,Min,Mid,Max,Median,Age_Embauche,Compa_Ratio,Range_Penetration,Market_Ratio,Ratio_Grille_Marche,Amplitude_Grille
0,Mat00001,NaN,Pole 3,P3-E13,Maroc,RETAIL BANKING,JT1389,5,216.40,46,...,142.67,212.94,283.21,212.94,46,1.016249,0.524619,1.016249,1.0,0.659998
1,Mat00002,NaN,Pole 3,P3-E06,Maroc,RETAIL BANKING,JT1109,3,304.48,40,...,241.11,359.87,478.63,359.87,40,0.846083,0.266799,0.846083,1.0,0.660016
2,Mat00003,NaN,Pole 3,P3-E01,Maroc,RETAIL BANKING,JT1103,2,544.96,35,...,313.45,467.83,622.21,467.83,33,1.164868,0.749806,1.164868,1.0,0.659983
3,Mat00004,NaN,Pole 4,P4-E03,Maroc,FINANCE,JT0355,4,331.99,33,...,185.47,276.82,368.17,276.82,24,1.199299,0.801970,1.199299,1.0,0.659996
4,Mat00005,NaN,Pole 4,P4-E05,Maroc,IT,JT0939,7,197.78,30,...,134.00,200.00,266.00,200.00,25,0.988900,0.483182,0.988900,1.0,0.660000


---
## Se servir des graphes

Tous les graphes qui suivent sont interactifs :
* **Zoomer** : tourner la molette de la souris sur le graphe, ou dessiner un rectangle avec la souris autour de la zone à agrandir.
* **Se déplacer** : cliquer sur la croix fléchée (en haut à droite du graphe), puis faire glisser le graphe.
* **Étirer un seul axe** : faire glisser le bout d'un axe (près du premier ou du dernier chiffre). Faire glisser le milieu d'un axe le décale.
* **Revenir à la vue de départ** : double-cliquer sur le graphe.
* **Changer d'échelle** : boutons « Échelle normale » et « Échelle logarithmique » au-dessus du graphe. En échelle logarithmique, passer de 100 à 200 prend la même place que passer de 1 000 à 2 000, parce que c'est le même doublement. Ça sert quand quelques très grandes valeurs écrasent le reste : les petites valeurs redeviennent lisibles.
* **Savoir à qui correspond un point** : passer la souris dessus. Le matricule, le nom, le poste, le grade et le salaire de la personne s'affichent.
* **Cacher une courbe ou un groupe** : cliquer sur son nom dans la légende. Double-cliquer dessus pour n'afficher que lui.
* **Enregistrer le graphe en image** (pour le support du comité) : cliquer sur l'appareil photo, en haut à droite. L'image garde le zoom en cours.

In [ ]:
# Réglages communs à tous les graphes interactifs

# Colonnes affichées quand la souris passe sur un salarié (seulement celles présentes dans le fichier)
INFOS_SURVOL = [c for c in ['Matricule', 'Nom', 'Job_Title', 'Grade', 'Fixe_Annuel_MAD'] if c in df.columns]

# scrollZoom : la molette zoome directement, sans choisir d'outil avant.
# toImageButtonOptions : l'appareil photo enregistre une image 2 fois plus fine, qui reste nette dans le support du comité.
CONFIG = {'scrollZoom': True, 'displaylogo': False, 'toImageButtonOptions': {'format': 'png', 'scale': 2}}


def montrer(fig, hauteur=600, axes_echelle=('yaxis',)):
    """Affiche le graphe avec les réglages communs.
    axes_echelle : les axes verticaux que les boutons « Échelle » passent en logarithmique
    ('yaxis' = axe du 1er graphique, 'yaxis2' = axe du 2e...). Vide : pas de boutons."""
    marge_haut, marge_bas = 130, 90
    if axes_echelle:
        def echelle(type_axe):
            # autorange : après le changement d'échelle, le graphe se recadre sur toutes les données
            return [{f'{axe}.{reglage}': valeur for axe in axes_echelle
                     for reglage, valeur in [('type', type_axe), ('autorange', True)]}]
        fig.update_layout(updatemenus=[dict(
            type='buttons', direction='right', x=0, xanchor='left', yanchor='bottom',
            # La hauteur se donne en part de la zone de dessin : ce calcul garde les boutons
            # 60 pixels au-dessus du dessin, au-dessus des titres, quelle que soit la hauteur du graphe
            y=1 + 60 / (hauteur - marge_haut - marge_bas),
            buttons=[dict(label='Échelle normale', method='relayout', args=echelle('linear')),
                     dict(label='Échelle logarithmique', method='relayout', args=echelle('log'))])])
    # Légende sous le graphe, à l'horizontale : à droite, elle prendrait la largeur des graphiques
    fig.update_layout(template='plotly_white', height=hauteur, margin=dict(t=marge_haut, b=marge_bas),
                      legend=dict(orientation='h', x=0, y=-0.15, yanchor='top'))
    fig.show(config=CONFIG)


def histogramme_et_courbe(serie, couleur, nb_barres='auto'):
    """Barres de l'histogramme + courbe lissée (comme kde=True de seaborn), à la même hauteur que les barres."""
    serie = serie.dropna()
    effectifs, bords = np.histogram(serie, bins=nb_barres)
    largeur = bords[1] - bords[0]
    barres = go.Bar(x=(bords[:-1] + bords[1:]) / 2, y=effectifs, width=largeur, marker_color=couleur, opacity=0.6,
                    showlegend=False, customdata=np.column_stack([bords[:-1], bords[1:]]),
                    hovertemplate='De %{customdata[0]:.2f} à %{customdata[1]:.2f} : %{y} salariés<extra></extra>')
    x = np.linspace(serie.min(), serie.max(), 300)
    # La courbe lissée a une surface de 1 : on la multiplie par le nombre de salariés et la largeur
    # d'une barre pour qu'elle épouse le haut des barres
    courbe = go.Scatter(x=x, y=gaussian_kde(serie)(x) * len(serie) * largeur, mode='lines',
                        line=dict(color=couleur, width=2), showlegend=False, hoverinfo='skip')
    return barres, courbe

---
## Visualisation 1 : Distribution de la Variable Cible (`Fixe_Annuel_MAD` vs Transformation Logarithmique)

### Objectif analytique
Les structures salariales présentent généralement une asymétrie vers les hautes rémunérations : la majorité des effectifs se concentre sur les grades opérationnels et intermédiaires, tandis que les fonctions de direction étirent la distribution vers la droite. Or, un modèle de régression linéaire standard est sensible aux valeurs extrêmes.

### Grille de lecture
* **Graphique de gauche (Salaire brut en kMAD) :** Permet d'évaluer l'étalement de la distribution et le poids des hauts salaires (points situés au-delà de la moustache supérieure).
* **Graphique de droite (Logarithme du salaire) :** Permet de vérifier si le passage au logarithme normalise la distribution (courbe en cloche symétrique, coefficient d'asymétrie proche de zéro).

### Arbitrage de modélisation
* Si la distribution logarithmique est plus symétrique, nous avons tout intérêt à modéliser `Log(Fixe_Annuel_MAD)`.
* **Intérêt métier :** En prédisant le logarithme du salaire, le modèle raisonne en **variations relatives (%)** (par exemple : une année d'ancienneté ou un niveau de performance supplémentaire génère +X % de rémunération)

**Passer au logarithme** nous permet de raisonner en pourcentage (%) plutôt qu'en dirhams bruts, ce qui rééquilibre la courbe et évite que les très hauts salaires ne faussent le modèle.

In [2]:
log_sal = np.log(df['Fixe_Annuel_MAD'])
# px a besoin d'une colonne pour afficher, au survol, les infos du salarié derrière chaque point
df_graphe = df.assign(Log_Fixe_Annuel_MAD=log_sal)

fig = make_subplots(rows=2, cols=2, row_heights=[0.75, 0.25], shared_xaxes=True, vertical_spacing=0.05,
                    subplot_titles=[f"Distribution de Fixe_Annuel_MAD<br>(Coefficient d'asymetrie = {df['Fixe_Annuel_MAD'].skew():.2f})",
                                    f"Distribution de Log(Fixe_Annuel_MAD)<br>(Coefficient d'asymetrie = {log_sal.skew():.2f})", '', ''])

for col, (colonne, couleur) in enumerate([('Fixe_Annuel_MAD', 'royalblue'), ('Log_Fixe_Annuel_MAD', 'seagreen')], start=1):
    # En haut : histogramme et courbe lissee
    for trace in histogramme_et_courbe(df_graphe[colonne], couleur):
        fig.add_trace(trace, row=1, col=col)
    # En bas : boite a moustaches ; la souris sur un point isole dit de quel salarie il s'agit
    for trace in px.box(df_graphe, x=colonne, points='outliers', hover_data=INFOS_SURVOL, color_discrete_sequence=[couleur]).data:
        fig.add_trace(trace, row=2, col=col)
    fig.update_xaxes(title_text=colonne, row=2, col=col)

fig.update_yaxes(title_text='Nombre de salaries', row=1)
fig.update_yaxes(showticklabels=False, row=2)
montrer(fig, hauteur=650, axes_echelle=())

print("Indicateur de reference : plus le coefficient d'asymetrie (skewness) est proche de 0, plus la distribution est symetrique.")

Indicateur de reference : plus le coefficient d'asymetrie (skewness) est proche de 0, plus la distribution est symetrique.


en passant en pourcentage (Log), l'échelle compresse les grands écarts en dirhams du haut de la pyramide (donc beaucoup moins d'outliers à droite) et met en évidence les petits salaires qui sont, en proportion, trois à quatre fois plus bas que le cœur de l'entreprise (d'où les quelques outliers qui apparaissent à gauche)

---
## Visualisation 2 : Alignement des Rémunérations avec l'Architecture des Grades, les Grilles Internes et le Marché

### Objectif analytique
Cette double visualisation confronte les salaires réellement pratiqués aux repères de notre politique de rémunération (`Min`, `Mid`, `Max`) ainsi qu'à la médiane du marché (`Median`), grade par grade.

### Grille de lecture
1. **Progression par Grade (Graphique de gauche) :** On observe la pente de progression salariale d'un grade à l'autre ainsi que l'écart éventuel entre notre milieu de grille (`Mid`, courbe bleue) et le marché (`Median`, courbe rouge).
2. **Respect des bornes salariales (Graphique de droite - Range Penetration) :**
   * Une valeur inférieure à `0.0` signale les collaborateurs rémunérés sous le minimum de leur bande (*Green Circles*).
   * Une valeur égale à `0.5` correspond exactement au milieu de grille (`Mid`).
   * Une valeur supérieure à `1.0` identifie les collaborateurs dépassant le plafond de leur grade (*Red Circles*).

### Arbitrage de modélisation
* Le milieu de grille (`Mid`) et la médiane marché (`Median`) synthétisent déjà le croisement entre le niveau de responsabilité (`Grade`) et le métier (`Job_Family`). Ils constitueront le socle d'ancrage de la régression.
* L'examen des collaborateurs hors bornes nous permet de décider si certains cas extrêmes relèvent d'anomalies ponctuelles à écarter de l'échantillon d'apprentissage pour ne pas biaiser l'estimation.

In [3]:
grade_order = sorted(df['Grade'].dropna().unique())
ref_grade = df.groupby('Grade')[['Min', 'Mid', 'Max', 'Median']].mean().reindex(grade_order).reset_index()

fig = make_subplots(rows=1, cols=2, subplot_titles=['Salaires Reels vs Grille Interne (Mid)<br>et Marche (Median) par Grade',
                                                    'Distribution de la Penetration<br>dans les Bandes Salariales'])

# Graphique 1 : Progression par Grade face a la grille et au marche
# Bande [Min - Max] en premier, pour qu'elle passe sous les boites : on trace le Max, puis le Min colorie jusqu'au Max
fig.add_trace(go.Scatter(x=ref_grade['Grade'], y=ref_grade['Max'], mode='lines', line_width=0,
                         name='Max moyen', showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=ref_grade['Grade'], y=ref_grade['Min'], mode='lines', line_width=0, fill='tonexty',
                         fillcolor='rgba(0, 0, 255, 0.1)', name='Amplitude Grille [Min - Max]'), row=1, col=1)
for trace in px.box(df, x='Grade', y='Fixe_Annuel_MAD', points='outliers', hover_data=INFOS_SURVOL,
                    color_discrete_sequence=['gray']).data:
    fig.add_trace(trace.update(name='Salaires reels', showlegend=True), row=1, col=1)
fig.add_trace(go.Scatter(x=ref_grade['Grade'], y=ref_grade['Mid'], mode='lines+markers',
                         line=dict(color='blue', width=2.5), name='Milieu de Grille moyen (Mid)'), row=1, col=1)
fig.add_trace(go.Scatter(x=ref_grade['Grade'], y=ref_grade['Median'], mode='lines+markers', marker_symbol='square',
                         line=dict(color='crimson', width=2.5, dash='dash'), name='Mediane Marche moyenne (Median)'), row=1, col=1)
# 'category' : chaque grade prend la meme largeur, meme si des numeros de grade manquent
fig.update_xaxes(type='category', categoryorder='array', categoryarray=grade_order, title_text='Grade', row=1, col=1)
fig.update_yaxes(title_text='Fixe_Annuel_MAD', row=1, col=1)

# Graphique 2 : Penetration de bande salariale
for trace in histogramme_et_courbe(df['Range_Penetration'], 'slateblue', nb_barres=40):
    fig.add_trace(trace, row=1, col=2)
for x, couleur, trait, texte, position in [(0.0, 'darkorange', 'dash', 'Min (0%)', 'top left'),
                                           (0.5, 'blue', 'solid', 'Mid (50%)', 'top right'),
                                           (1.0, 'crimson', 'dash', 'Max (100%)', 'top right')]:
    fig.add_vline(x=x, line=dict(color=couleur, dash=trait, width=2), annotation_text=texte,
                  annotation_position=position, row=1, col=2)
fig.update_xaxes(title_text='Range Penetration : 0 = Min | 0.5 = Mid | 1.0 = Max', row=1, col=2)
fig.update_yaxes(title_text='Nombre de salaries', row=1, col=2)

montrer(fig, axes_echelle=('yaxis', 'yaxis2'))

sous_min = (df['Fixe_Annuel_MAD'] < df['Min']).sum()
sur_max = (df['Fixe_Annuel_MAD'] > df['Max']).sum()
print(f"Synthese : {sous_min} collaborateurs sous le Min ({sous_min/len(df)*100:.1f}%) | {sur_max} collaborateurs au-dessus du Max ({sur_max/len(df)*100:.1f}%).")

Synthese : 250 collaborateurs sous le Min (2.5%) | 150 collaborateurs au-dessus du Max (1.5%).


---
## Visualisation 3 : Analyse par Famille de Métier (`Job_Family`) et Compétitivité des Grilles

### Objectif analytique
Bien que chaque famille de métier dispose de sa propre grille (`Mid`), nous cherchons ici à vérifier si certaines filières métiers se positionnent systématiquement plus haut dans leur bande salariale que d'autres, et si ce phénomène s'explique par un décalage entre notre grille interne et la réalité du marché (`Mid / Median`).

### Grille de lecture
* **Graphique supérieur (`Compa-Ratio = Salaire / Mid`) :** Mesure le positionnement réel des équipes par rapport au milieu de leur grille interne (cible théorique = `1.0`).
* **Graphique inférieur (`Ratio_Grille_Marche = Mid / Median`) :** Mesure la compétitivité faciale de nos grilles par rapport au marché externe. Lorsqu'une grille interne est en retrait face au marché (`Mid / Median < 1.0`), la pression au recrutement pousse souvent les salaires réels vers le haut de la grille (`Compa-Ratio > 1.0`).

### Arbitrage de modélisation
* Si des écarts de Compa-Ratio apparaissent entre les familles de métiers, l'inclusion conjointe de `Job_Family` et du ratio structurel `Ratio_Grille_Marche` (`Mid / Median`) dans la régression est indispensable pour capter ces pratiques différenciées.

In [4]:
order_jf = list(df.groupby('Job_Family')['Compa_Ratio'].median().sort_values(ascending=False).index)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1,
                    subplot_titles=['Positionnement Salarial Interne (Compa-Ratio = Salaire / Mid) par Famille de Metier',
                                    'Competitivite des Grilles Internes (Ratio Mid / Median) par Famille de Metier'])

# Graphique 1 : Compa-Ratio par famille de metier ; Graphique 2 : competitivite de la grille interne face au marche.
# Un metier garde la meme couleur dans les deux graphiques : on le suit facilement de haut en bas.
for ligne, colonne in [(1, 'Compa_Ratio'), (2, 'Ratio_Grille_Marche')]:
    for trace in px.box(df, x='Job_Family', y=colonne, color='Job_Family', category_orders={'Job_Family': order_jf},
                        color_discrete_sequence=px.colors.qualitative.Set2, points='outliers', hover_data=INFOS_SURVOL).data:
        fig.add_trace(trace.update(showlegend=False), row=ligne, col=1)
    fig.update_yaxes(title_text=colonne, row=ligne, col=1)

# Les lignes de repere sont nommees dans la legende (un texte pose sur le graphe cacherait des boites)
fig.add_hline(y=1.0, line=dict(color='red', dash='dash', width=2), name='Cible Milieu de Grille (Compa-Ratio = 1.0)', showlegend=True, row=1, col=1)
fig.add_hline(y=1.0, line=dict(color='black', dash='dash', width=2), name='Parite Grille = Marche (Mid = Median)', showlegend=True, row=2, col=1)
fig.update_xaxes(categoryorder='array', categoryarray=order_jf)
fig.update_xaxes(tickangle=-45, row=2, col=1)
# 'overlay' : une seule boite par metier, centree sur son nom (sinon plotly les decale comme pour des groupes)
fig.update_layout(boxmode='overlay')

montrer(fig, hauteur=900, axes_echelle=('yaxis', 'yaxis2'))

---
## Visualisation 4 : Séniorité (`Age`, `Anciennete`) et Détection de la Compression Salariale

### Objectif analytique
L'analyse conjointe de l'âge et de l'ancienneté répond à deux enjeux majeurs :
1. **Traitement de la colinéarité :** L'âge et l'ancienneté évoluent mécaniquement dans le même sens. Leur introduction simultanée à l'état brut dégrade la précision des coefficients de régression. En créant la variable `Age_Embauche = Age - Anciennete`, nous isolons d'un côté l'expérience acquise *antérieurement à l'embauche*, et de l'autre la séniorité acquise *en interne*.
2. **Analyse des rendements d'ancienneté et de la compression :** La courbe de tendance non-linéaire (*Lowess*) permet de vérifier si la progression dans la grille ralentit après un certain nombre d'années ou s'il existe un phénomène de compression salariale (nouveaux recrutés embauchés à des niveaux supérieurs aux collaborateurs ayant 2 à 4 ans d'ancienneté).

### Grille de lecture
* **Graphique de gauche (`Anciennete` vs `Compa-Ratio`) :** Une inflexion ou un plafonnement de la courbe rouge indique un rendement marginal décroissant de l'ancienneté. Une pente initiale négative entre 0 et 3 ans signale une compression salariale à l'embauche.
* **Graphique de droite (`Age_Embauche` vs `Compa-Ratio`) :** Évalue comment l'expérience externe préalable est valorisée dans la bande salariale.

### Arbitrage de modélisation
* Substitution du couple `(Age, Anciennete)` par le couple orthogonal `(Age_Embauche, Anciennete)`.
* Ajout d'un terme quadratique (`Anciennete²`) dans la régression si la courbe de gauche présente un plafonnement marqué.

Le graphe pose donc la question : à poste égal, est-on mieux placé dans sa grille selon l'ancienneté (à gauche) ou selon l'âge à l'embauche (à droite) ?

In [5]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["Progression dans la Grille (Compa-Ratio)<br>selon l'Anciennete",
                                                    "Valorisation de l'Experience Externe (Age a l'embauche)<br>sur le Compa-Ratio"])

# Graphique 1 : Anciennete vs Compa-Ratio ; Graphique 2 : Age a l'embauche vs Compa-Ratio
for col, (x, couleur_points, couleur_courbe, titre_x) in enumerate([
        ('Anciennete', 'teal', 'crimson', 'Anciennete (annees)'),
        ('Age_Embauche', 'purple', 'darkorange', "Age a l'embauche (Age - Anciennete)")], start=1):
    # trendline='lowess' : la meme courbe de tendance non-lineaire que regplot(lowess=True)
    nuage = px.scatter(df, x=x, y='Compa_Ratio', opacity=0.25, hover_data=INFOS_SURVOL, trendline='lowess',
                       color_discrete_sequence=[couleur_points], trendline_color_override=couleur_courbe)
    points, courbe = nuage.data
    fig.add_trace(points, row=1, col=col)
    fig.add_trace(courbe.update(name=f'Tendance non-lineaire (Lowess) : {x}', showlegend=True, line_width=3), row=1, col=col)
    fig.add_hline(y=1.0, line=dict(color='black', dash='dash'), row=1, col=col)
    fig.update_xaxes(title_text=titre_x, row=1, col=col)
    fig.update_yaxes(title_text='Compa-Ratio (Salaire / Mid)', row=1, col=col)

montrer(fig, axes_echelle=('yaxis', 'yaxis2'))

print(f"Correlation lineaire brute Age / Anciennete         : {df['Age'].corr(df['Anciennete']):.2f}")
print(f"Correlation apres retraitement Age_Embauche / Anciennete : {df['Age_Embauche'].corr(df['Anciennete']):.2f}")

Correlation lineaire brute Age / Anciennete         : 0.09
Correlation apres retraitement Age_Embauche / Anciennete : -0.37


---
## Visualisation 5 : Différenciation par la Performance (`Competence_N1`, `Positionnement_9BOX`) et la Tension Métier (`Hot_job`)

### Objectif analytique
Nous vérifions ici dans quelle mesure les éléments individuels de performance/potentiel (`Competence_N1`, `Positionnement_9BOX`) et la rareté du poste sur le marché (`Hot_job`) se traduisent par une différenciation salariale effective au sein des grilles.

### Grille de lecture
* **Matrice de gauche (Croisement `Positionnement_9BOX` x `Hot_job`) :** Indique le Compa-Ratio moyen pour chaque segment. Une politique de rétention ciblée se traduit par des valeurs plus élevées dans le quadrant supérieur droit (fort potentiel/performance sur un métier en forte tension, `Hot_job = 2`).
* **Graphique de droite (`Competence_N1` vs `Compa-Ratio`) :** La pente des droites mesure la sensibilité du salaire à l'évaluation des compétences de l'année précédente, selon le niveau de tension du métier.

### Arbitrage de modélisation
* Intégration de `Competence_N1`, `Positionnement_9BOX` et `Hot_job`.
* Si la matrice de gauche révèle un effet multiplicatif sur les profils à haut potentiel occupant des postes critiques, nous intégrerons un terme d'interaction explicite (`Positionnement_9BOX * Hot_job`).

In [14]:
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.15,
                    subplot_titles=['Compa-Ratio Moyen : Croisement<br>Positionnement 9-Box x Tension Metier (Hot_job)',
                                    'Relation entre la Note de Competence_N1 et le Compa-Ratio<br>selon la Tension (Hot_job)'])

# Graphique 1 : Matrice Compa-Ratio moyen par 9-Box et Hot_job
# Pas de tri a l'envers : plotly met la premiere ligne en bas, donc la plus haute note 9-Box est deja en haut
pivot_9box = df.pivot_table(index='Positionnement_9BOX', columns='Hot_job', values='Compa_Ratio', aggfunc='mean')
# Le nombre de salaries de chaque case s'affiche au survol : une moyenne sur 3 personnes pese moins que sur 300
effectifs_9box = df.pivot_table(index='Positionnement_9BOX', columns='Hot_job', values='Compa_Ratio', aggfunc='count')
fig.add_trace(go.Heatmap(z=pivot_9box.values, x=pivot_9box.columns.astype(str), y=pivot_9box.index.astype(str),
                         customdata=effectifs_9box.values, colorscale='YlOrRd', texttemplate='%{z:.3f}',
                         colorbar=dict(title='Compa-Ratio Moyen', x=0.43),
                         hovertemplate='9-Box %{y} / Hot_job %{x}<br>Compa-Ratio moyen : %{z:.3f}<br>Nombre de salaries : %{customdata}<extra></extra>'),
              row=1, col=1)
fig.update_xaxes(type='category', title_text='Tension du metier Hot_job (0 = Faible a 2 = Forte)', row=1, col=1)
fig.update_yaxes(type='category', title_text='Positionnement 9-Box (0 a 6)', row=1, col=1)

# Graphique 2 : Competence N-1 vs Compa-Ratio par niveau de tension, avec une droite de tendance par niveau
# Ces couleurs sont celles que seaborn affichait (la liste de l'ancien code n'etait pas utilisee, et son jaune se lisait mal sur fond blanc)
df_graphe = df.assign(Tension=df['Hot_job'].map(lambda v: f'Hot_job = {v}'))
ordre_tension = [f'Hot_job = {v}' for v in sorted(df['Hot_job'].dropna().unique())]
for trace in px.scatter(df_graphe, x='Competence_N1', y='Compa_Ratio', color='Tension', category_orders={'Tension': ordre_tension},
                        color_discrete_sequence=['#4878d0', '#ee854a', '#6acc64'], opacity=0.15, trendline='ols',
                        hover_data=INFOS_SURVOL).data:
    fig.add_trace(trace, row=1, col=2)
# Droites de tendance plus epaisses, pour qu'elles ressortent au milieu des points
fig.update_traces(line_width=3, selector=dict(mode='lines'))
fig.add_hline(y=1.0, line=dict(color='black', dash='dash'), row=1, col=2)
fig.update_xaxes(title_text='Competence_N1', row=1, col=2)
fig.update_yaxes(title_text='Compa-Ratio', row=1, col=2)
fig.update_layout(legend_title_text='Tension Hot_job')

montrer(fig, axes_echelle=('yaxis2',))

---
## Visualisation 6 : Analyse de l'Équité Interne (`Sexe`) et des Pratiques par Pôle (`Entite_N1`)

### Objectif analytique
1. **Équité professionnelle (`Sexe`) :** Une comparaison globale des salaires moyens entre hommes et femmes est souvent biaisée par des effets de structure (répartition par grade ou par filière métier). La comparaison du `Compa_Ratio` à grade identique permet d'isoler l'écart résiduel à périmètre de responsabilité comparable.
2. **Effet organisationnel (`Entite_N1`) :** L'examen du `Compa-Ratio` par Pôle neutralise la structure des emplois et met en lumière d'éventuelles différences d'historique budgétaire entre les entités.

### Grille de lecture
* **Graphique de gauche :** Alignement des distributions F et M au sein de chaque grade.
* **Graphique de droite :** Dispersion des médianes de Compa-Ratio entre les différents pôles (`Entite_N1`).

### Arbitrage de modélisation (Point d'arbitrage managérial)
* **Variable `Entite_N1` :** À inclure (encodage catégoriel) si des écarts significatifs entre pôles sont constatés.
* **Variable `Sexe` :** Deux approches selon la finalité que nous donnons au modèle :
  * *Finalité d'audit d'équité :* Inclusion de la variable `Sexe` dans la régression afin de mesurer statistiquement l'existence ou l'absence d'écart salarial toutes choses égales par ailleurs.
  * *Finalité de recommandation salariale cible :* Exclusion de la variable `Sexe` du modèle prédictif afin de garantir une neutralité absolue des prédictions.

In [7]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["Analyse d'Equite : Compa-Ratio F vs M<br>a Grade Equivalent",
                                                    'Positionnement Salarial par Pole (Entite_N1)<br>a Grille Neutralisee'])

# Graphique 1 : Equite F/M par Grade (points=False : pas de points isoles, comme showfliers=False)
for trace in px.box(df, x='Grade', y='Compa_Ratio', color='Sexe', points=False, category_orders={'Sexe': sorted(df['Sexe'].dropna().unique())},
                    color_discrete_sequence=['#4878d0', '#ee854a']).data:
    fig.add_trace(trace, row=1, col=1)
fig.add_hline(y=1.0, line=dict(color='red', dash='dash'), row=1, col=1)
fig.update_xaxes(type='category', categoryorder='array', categoryarray=grade_order, title_text='Grade', row=1, col=1)

# Graphique 2 : Positionnement par Pole (Entite_N1)
order_entite = list(df.groupby('Entite_N1')['Compa_Ratio'].median().sort_values(ascending=False).index)
for trace in px.box(df, x='Entite_N1', y='Compa_Ratio', points=False, color_discrete_sequence=['steelblue']).data:
    fig.add_trace(trace, row=1, col=2)
fig.add_hline(y=1.0, line=dict(color='red', dash='dash'), name='Milieu de Grille (1.0)', showlegend=True, row=1, col=2)
fig.update_xaxes(categoryorder='array', categoryarray=order_entite, tickangle=-30, title_text='Entite_N1', row=1, col=2)

fig.update_yaxes(title_text='Compa_Ratio', row=1)
# 'group' : les boites F et M d'un meme grade se placent cote a cote
fig.update_layout(boxmode='group', legend_title_text='Sexe')

montrer(fig, axes_echelle=('yaxis', 'yaxis2'))

ecart_brut = (df[df['Sexe']=='M']['Fixe_Annuel_MAD'].mean() / df[df['Sexe']=='F']['Fixe_Annuel_MAD'].mean() - 1) * 100
ecart_compa = (df[df['Sexe']=='M']['Compa_Ratio'].mean() / df[df['Sexe']=='F']['Compa_Ratio'].mean() - 1) * 100
print(f"Ecart salarial moyen non corrige (M vs F)                     : {ecart_brut:+.2f}%")
print(f"Ecart moyen a grille et metier constants (Compa-Ratio M vs F) : {ecart_compa:+.2f}%")

Ecart salarial moyen non corrige (M vs F)                     : -0.17%
Ecart moyen a grille et metier constants (Compa-Ratio M vs F) : -0.37%


---
## Visualisation 7 : Contrôle de la Multicolinéarité (Matrice de Corrélation et Facteur d'Inflation de la Variance - VIF)

### Objectif analytique
Avant d'estimer l'équation de régression, nous devons vérifier l'absence de redondance forte entre les variables explicatives (multicolinéarité). En particulier, les bornes de grille (`Min`, `Mid`, `Max`), la médiane marché (`Median`) et le `Grade` évoluent de concert. L'indicateur statistique de référence est le **VIF (*Variance Inflation Factor*)** :
* **VIF < 5 :** Absence de colinéarité problématique, les estimations des coefficients sont stables.
* **VIF > 10 :** Forte redondance statistique qui fausse la lecture individuelle des coefficients.

### Grille de lecture
* **Premier tableau VIF (Variables brutes) :** Met en évidence la redondance mécanique entre `Min`, `Mid`, `Max`, `Median`, `Grade`, `Age` et `Anciennete`.
* **Second tableau VIF (Sélection retraitée) :** Démontre qu'en conservant `Mid` comme point d'ancrage, complété par le ratio relatif `Ratio_Grille_Marche` (`Mid / Median`) et par `Age_Embauche`, tous les indicateurs VIF passent sous le seuil de tolérance (< 5).

### Arbitrage de modélisation
* Validation du périmètre retraité (second tableau) qui préserve l'intégralité de l'information économique tout en garantissant l'interprétabilité des poids de chaque variable.

In [15]:
# 1. Matrice de correlation lineaire
cols_corr = ['Fixe_Annuel_MAD', 'Grade', 'Mid', 'Min', 'Max', 'Median',
             'Ratio_Grille_Marche', 'Age', 'Age_Embauche', 'Anciennete',
             'Competence_N1', 'Positionnement_9BOX', 'Hot_job']

# RdBu_r : bleu pour les correlations negatives, rouge pour les positives, blanc autour de 0 (comme coolwarm)
fig = px.imshow(df[cols_corr].corr(), text_auto='.2f', color_continuous_scale='RdBu_r', zmin=-1, zmax=1, aspect='auto',
                title='Matrice de Correlation des Variables Numeriques et Ordinales')
montrer(fig, hauteur=750, axes_echelle=())

# 2. Calcul comparatif des VIF
def calc_vif(data, cols):
    X = data[cols].dropna().copy()
    # Une colonne qui ne varie pas (ex. Ratio_Grille_Marche quand la grille est egale au marche) n'apporte
    # aucune information, et statsmodels la prend pour la constante : on l'ecarte et on le signale
    constantes = [col for col in X.columns if X[col].nunique() <= 1]
    if constantes:
        print(f"Colonnes sans variation, ecartees du calcul VIF : {constantes}")
        X = X.drop(columns=constantes)
    # has_constant='add' : la constante est toujours ajoutee, donc toujours en position 0
    X_const = sm.add_constant(X, has_constant='add')
    vif = pd.DataFrame()
    vif['Variable'] = X.columns
    vif['VIF'] = [variance_inflation_factor(X_const.values, i+1) for i in range(len(X.columns))]
    return vif.sort_values(by='VIF', ascending=False)

cols_brutes = ['Grade', 'Min', 'Mid', 'Max', 'Median', 'Age', 'Anciennete', 'Competence_N1', 'Positionnement_9BOX', 'Hot_job']
cols_propres = ['Mid', 'Ratio_Grille_Marche', 'Age_Embauche', 'Anciennete', 'Competence_N1', 'Positionnement_9BOX', 'Hot_job']

print("1. DIAGNOSTIC VIF SUR VARIABLES BRUTES (Colinearite elevee) :")
display(calc_vif(df, cols_brutes))

print("\n2. DIAGNOSTIC VIF SUR VARIABLES RETRAITEES (Stabilite statistique, VIF < 5) :")
display(calc_vif(df, cols_propres))

1. DIAGNOSTIC VIF SUR VARIABLES BRUTES (Colinearite elevee) :


,Variable,VIF
4,Median,1.000800e+15
2,Mid,1.000800e+15
3,Max,1.219863e+10
1,Min,3.533876e+09
0,Grade,8.667843e+00
9,Hot_job,1.008507e+00
6,Anciennete,1.007938e+00
5,Age,1.007936e+00
8,Positionnement_9BOX,1.001924e+00
7,Competence_N1,1.000942e+00



2. DIAGNOSTIC VIF SUR VARIABLES RETRAITEES (Stabilite statistique, VIF < 5) :
Colonnes sans variation, ecartees du calcul VIF : ['Ratio_Grille_Marche']


,Variable,VIF
1,Age_Embauche,1.159847
2,Anciennete,1.159761
4,Positionnement_9BOX,1.000846
0,Mid,1.000839
3,Competence_N1,1.000713
5,Hot_job,1.000591
